In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from sklearn.metrics import r2_score

# 三个文件路径
file_paths = [
    "E:/Desktop/hybir/419_pre.csv",
    "E:/Desktop/hybir/417_pre.csv",
    "E:/Desktop/hybir/414_pre.csv"
]

# 合并数据
all_exp, all_pre = [], []

for path in file_paths:
    df = pd.read_csv(path)
    all_exp.extend(df['exp'].values)
    all_pre.extend(df['pre'].values)

# 转为 numpy 数组
exp = np.array(all_exp)
pre = np.array(all_pre)

# 去除 <=0 的值以避免 log 错误
valid = (exp > 0) & (pre > 0)
exp = exp[valid]
pre = pre[valid]

# log10 转换
log_exp = np.log10(exp)
log_pre = np.log10(pre)

# 回归拟合
slope, intercept, r_value, p_value, std_err = linregress(log_exp, log_pre)
r_squared = r_value**2

# 原始空间 R²
r2_raw = r2_score(exp, pre)

# ==== 校正预测值 ====
# 把 log_pre 校正到拟合线（即 log_exp 对应的拟合值）
log_pre_corrected = slope * log_exp + intercept

# 反变换回原始尺度
pre_corrected = 10 ** log_pre_corrected

# 校正后的 R²（更体现拟合线效果）
r2_corrected = r2_score(exp, pre_corrected)

# 可视化（log-log 空间）
fig, ax = plt.subplots(figsize=(5.08, 5.08))
ax.scatter(log_exp, log_pre, s=10, alpha=0.7)
ax.plot(log_exp, log_pre_corrected, color='green', linewidth=2)

ax.set_xlabel("log10(exp)", fontsize=10)
ax.set_ylabel("log10(pre)", fontsize=10)
ax.set_title('hybrid', fontsize=12)

# 标注
#ax.annotate(f"Pearson R² = {r_squared:.4f}", xy=(0.05, 0.92), xycoords='axes fraction', fontsize=8)
#ax.annotate(f"Raw R² = {r2_raw:.4f}", xy=(0.05, 0.85), xycoords='axes fraction', fontsize=8)
ax.annotate(f"R² = {r2_corrected:.4f}", xy=(0.05, 0.85), xycoords='axes fraction', fontsize=10)

# 美化边框
for spine in ax.spines.values():
    spine.set_linewidth(0.5)

plt.tight_layout()
plt.savefig("E:/Desktop/hybir/hybrid_R².svg")
plt.show()

# 控制台输出
print(f"Pearson R（log-log 空间）: {r_value:.4f}")
print(f"Pearson R²（log-log 空间）: {r_squared:.4f}")
print(f"原始空间 R²: {r2_raw:.4f}")
print(f"回归校正后 R²: {r2_corrected:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

def promoter(m1, m2):
    L1= m1
    L2 =m2
    #I1 = m1
    #I2 = m2
    k11 = 0.000268715
    k21 = 0.007083518
    k31 = 0.760771692
    #kx1 = 1.086833596
    #kx2 = 4.230793476
    kd1 = 0.745553315
    k12 = 0.000171923
    k22 = 0.256016091
    k32 = 225.0817618
    kd2 = 40.97704697
    Imax1 = 35.84931505
    Imax2 = 0.85546578

    I01 = 0.006969286
    I02 = 0.011797972
    I1 = 10
    I2 = 10
    #L1= 1.5
    #L2= 1.5
 
    x1 = np.sqrt((k21*I1+1)*(k21*I1+1)+ 8 * L1 * (k21*k21*k31*I1*I1 + k11))

    F1 = np.log(L1 / 2) + np.log((x1 - (k21*I1+1)) / (x1 + (k21*I1+1))) + np.log(kd1)

    #x1 = np.sqrt((k21*I1+kx1+kx2*k21*I1+1)*(k21*I1+kx1+kx2*k21*I1+1)+ 8 * L1 * (k21*k21*k31*I1*I1 + k11+k11*kx1*kx1+k31*k21*k21*kx2*kx2*I1*I1))
    x2 = np.sqrt((k22 * I2 + 1) * (k22 * I2 + 1) + 8 * L2 * (k22 * k22 * k32 * I2 * I2 + k12))
    #F1 = np.log(L1 / 2) + np.log((x1 - (k21*I1+kx1+kx2*k21*I1+1)) / (x1 + (k21*I1+kx1+kx2*k21*I1+1))) + np.log(kd1)
    F2 = np.log(L2 / 2) + np.log((x2 - (k22 * I2 + 1)) / (x2 + (k22 * I2 + 1))) + np.log(kd2)
    P =  (Imax1 / (1 + np.exp(-F1))) * (Imax2 / (1 + np.exp(F2))) + I01

    a1=(1 / (1 + np.exp(-F1)))
    a2=(1 / (1 + np.exp(F2)))

    #print(a1,a2)

    return P


m1 = np.logspace(0, 1,1000)
m2 = np.logspace(0, 1, 1000)
M1, M2 = np.meshgrid(m1, m2)
plt.figure(figsize=(7,6))
fig, ax = plt.subplots(figsize=(7,6))

colors = [(0.0, '#C9E3AC'), (0.5, '#90BE6D'), (1.0, '#EA9010')]
cmap = LinearSegmentedColormap.from_list('custom cmap', colors)



cp = ax.pcolormesh(M1, M2, (promoter(M1, M2)), cmap='coolwarm',norm=LogNorm(vmin=0.005, vmax=0.3))
#, norm=LogNorm(vmin=0.005, vmax=0.5)

cbar = fig.colorbar(cp)
cbar.ax.set_ylabel('P')

# 设置横纵坐标轴标签及其缩放
ax.set_xlabel('inducer_activator(um)')
ax.set_ylabel('inducer_repressor(um)')
ax.set_xscale('log')
ax.set_yscale('log')


#plt.title("pre")
plt.tight_layout()
'''
df = pd.read_csv("E:\Desktop\hybir\\417_1.csv")
a = df["inducer1"]
b = df["inducer2"]
c = df["RPU"]
#plt.scatter(a,c,  cmap='coolwarm',s=20)
'''
plt.show()
#plt.savefig('E:\Desktop\\LBD_DBD\\LasR.svg')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import r2_score

# === 模型函数 ===
def promoter(m1, m2):
    I1 = m1
    I2 = m2
    k11 = 0.000268715


    k21 = 0.007083518

    k31 = 0.760771692
    #kx1 = 1.086833596
    #kx2 = 4.230793476
    kd1 = 0.745553315

    k12 = 0.000171923
    k22 = 0.256016091
    k32 = 225.0817618
    kd2 = 40.97704697



    Imax1 = 35.84931505
    Imax2 = 0.687512595
    I01 = 0.015930056

    I02 = 0.035485714


    L1 = 10
    L2 = 10
    x1 = np.sqrt((k21*I1+1)*(k21*I1+1)+ 8 * L1 * (k21*k21*k31*I1*I1 + k11))

    F1 = np.log(L1 / 2) + np.log((x1 - (k21*I1+1)) / (x1 + (k21*I1+1))) + np.log(kd1)


    #x1 = np.sqrt((k21*I1 + kx1 + kx2*k21*I1 + 1)**2 +
    #             8 * L1 * (k21**2 * k31 * I1**2 + k11 + k11*kx1**2 + k31*k21**2*kx2**2*I1**2))
    x2 = np.sqrt((k22 * I2 + 1)**2 +
                 8 * L2 * (k22**2 * k32 * I2**2 + k12))

    #F1 = np.log(L1 / 2) + np.log((x1 - (k21*I1 + kx1 + kx2*k21*I1 + 1)) / (x1 + (k21*I1 + kx1 + kx2*k21*I1 + 1))) + np.log(kd1)
    F2 = np.log(L2 / 2) + np.log((x2 - (k22 * I2 + 1)) / (x2 + (k22 * I2 + 1))) + np.log(kd2)
    P =(Imax1 / (1 + np.exp(-F1))) * (Imax2 / (1 + np.exp(F2))) + I01
    #P =   (Imax1 / (1 + np.exp(-F1))) *  (Imax2 * (1 - ((1 - (1 / (1 + np.exp(F2))))**2)) )  + I01

    return P

# === 数据读取 ===
df = pd.read_csv("E:/Desktop/hybir/419_4.csv")
a = df["inducer1"]
b = df["inducer2"]
c = df["RPU"]

# === 散点图：实验点 ===
fig, ax = plt.subplots(figsize=(5.08, 5.08))
ax.scatter(b, c, c='black', s=20, label='Experimental')

# === R²计算 ===
pre = promoter(a.values, b.values)
exp = c.values
r2_raw = r2_score(exp, pre)

# 原始R²标注
plt.annotate(f"$R^2$ = {r2_raw:.4f}", xy=(0.05, 0.90), xycoords='axes fraction', fontsize=10)
# 固定几个 inducer1（activator）值：
unique_inducer1 = np.unique(df["inducer1"].round(4))

# 对每个固定 inducer1，画出不同的 inducer2（横轴）
for i, inducer1_fixed in enumerate(unique_inducer1):
    inducer2_range = np.logspace(-4, 1, 1000)
    curve = promoter(np.full_like(inducer2_range, inducer1_fixed), inducer2_range)
    ax.plot(inducer2_range, curve, label=f'inducer1 = {inducer1_fixed:.2f}')


# === 图形设置 ===
ax.set_xlabel('inducer(μM)')
ax.set_ylabel('Output (RPU)')
ax.set_xscale('log')
ax.set_yscale('log')

ax.set_ylim(1e-3, 1e2)
ax.legend(fontsize=7, loc='best')
plt.title("419_10-10", fontsize=10)
plt.tight_layout()
plt.savefig("E:/Desktop/hybir/curve/419_4.svg")
plt.show()
